49. Készítsünk listát a raktáron lévő termékek összértékéről raktárkód, azon belül kategóriakód, majd mennyiségi egység szerinti bontásban! A lista jelenítse meg a részösszegeket és a végösszeget is! A listát szűrjük az 5-ös és 9-es azonosítójú kategóriára! A csoportosításnál a ROLLUP záradékot használjuk!

In [13]:
SELECT RAKTAR_KOD, 
        KAT_ID,
        MEGYS,
        SUM(LISTAAR*KESZLET) as 'Összérték'
from Termek
where KAT_ID in (5,9)
GROUP BY rollup (RAKTAR_KOD, KAT_ID, MEGYS)

Query was canceled by user

Total execution time: 00:00:52.776

50.Készítsünk listát a raktáron lévő termékek összértékéről raktárkód, azon belül kategóriakód, majd mennyiségi egység szerinti bontásban! A lista jelenítse meg a részösszegeket és a végösszeget is! A listát szűrjük az 5-ös és 9-es azonosítójú kategóriára! A csoportosításnál a CUBE záradékot használjuk!

In [14]:
SELECT RAKTAR_KOD, 
        KAT_ID,
        MEGYS,
        SUM(LISTAAR*KESZLET) as 'Összérték'
from Termek
where KAT_ID in (5,9)
GROUP BY cube (RAKTAR_KOD, KAT_ID, MEGYS)

Query was canceled by user

Total execution time: 00:00:00.002

51. Készítsünk listát az egyes ügyfelek átlagos életkoráról az ügyfél neme, illetve az ügyfél születési éve szerint csoportosítva!A listát szűrjük azon ügyfelekre, akik neve D-vel vagy E-vel kezdődik! (Az életkor legyen a születési évtől a jelenlegi évig eltelt évek száma)

In [15]:
SELECT szulev, nem,
AVG(YEAR(GETDATE())-SZULEV)
AS 'Átlagos életkor'
FROM Ugyfel
WHERE NEV LIKE 'E%' OR NEV LIKE 'D%' GROUP BY
GROUPING SETS((SZULEV),(NEM))

Query was canceled by user

Total execution time: 00:00:00.001

52. listázzuk, hogy melyik évben hány db terméket rendeltek meg! A lista megfelelően jelölve jelenítse meg a rendelések teljes összegét is!

In [16]:
SELECT
(
CASE GROUPING(YEAR(REND_DATUM))
WHEN 0 THEN CAST(YEAR(REND_DATUM)
AS nvarchar(4))
WHEN 1 THEN 'Összesen' END
)
AS ÉV,
COUNT(*) AS 'DB'
FROM Rendeles
GROUP BY ROLLUP(YEAR(REND_DATUM))

Query was canceled by user

Total execution time: 00:00:00.006

53. listázzuk, hogy naponta, azon belül fizetési mód szerint hány rendelés történt! A lista megfelelően jelölve jelenítse meg a részösszegeket és a végösszeget is!

In [17]:
SELECT

IIF(
    GROUPING(REND_DATUM)=1,'Összesen',
CAST(REND_DATUM AS nvarchar(10))
) AS 'Rendelés dátuma',
CASE GROUPING_ID(REND_DATUM, FIZ_MOD)
    WHEN 0 THEN FIZ_MOD
    WHEN 1 THEN '**Fizetési módok összesen**'
    WHEN 3 THEN 'Összesen' 
END as 'FIZ_MOD',
COUNT(*) as 'DB'
FROM Rendeles
GROUP BY ROLLUP(REND_DATUM, FIZ_MOD)

Query was canceled by user

Total execution time: 00:00:00.004

54. <span style="background-color: rgb(255, 255, 255); color: rgb(0, 0, 0); font-family: &quot;Open Sans&quot;, sans-serif; font-size: 14.44px;">Készítsünk listát arról, hogy ügyfelenként (LOGIN), azon belül szállítási módonként hány megrendelés történt!&nbsp;</span>   

a. A lista tartalmazza a részösszegeket és a végösszeget is!  
b. Használjuk a ROLLUP záradékot!

In [18]:
SELECT [LOGIN],
        SZALL_MOD,
        COUNT(*)
from Rendeles
GROUP BY rollup ([LOGIN],SZALL_MOD)

Query was canceled by user

Total execution time: 00:00:00.001

55. <span style="background-color: rgb(255, 255, 255); color: rgb(0, 0, 0); font-family: &quot;Open Sans&quot;, sans-serif; font-size: 14.44px;">Készítsünk listát a termékek számáról a következő csoportosítási szempontok szerint:</span>

<span style="background-color: rgb(255, 255, 255); color: rgb(0, 0, 0); font-family: &quot;Open Sans&quot;, sans-serif; font-size: 14.44px;">kategória azonosító, raktárkód, raktárkód+mennyiségi egység!</span>

a. A listát szűrjük azokra a csoportokra, ahol a termékek száma legalább 6!

In [19]:
SELECT KAT_ID,
        RAKTAR_KOD,
        MEGYS,
        COUNT(*) as 'db'
from Termek
GROUP by GROUPING SETS((KAT_ID),(RAKTAR_KOD),(RAKTAR_KOD,MEGYS))
HAVING COUNT(*)>5

Query was canceled by user

Total execution time: 00:00:00

```
56. Készítsünk listát az egyes termékkategóriákban lévő termékek számáról! 

```

a. Elég megjeleníteni a kategóriák azonosítóit és a darabszámokat!   
b. A lista megfelelően jelölve tartalmazza a végösszeget is!   
 c. Az oszlopokat nevezzük el értelemszerűen!   
 d. A listát rendezzük a darabszám szerint növekvő sorrendbe!

In [20]:
SELECT 
(
case GROUPING(KAT_ID)
    when 0 then cast(KAT_ID as nvarchar(50))
    when 1 then 'Összesen'
end 
) as 'kategória',
    COUNT(*) as 'db'
from Termek
group BY rollup(KAT_ID)
ORDER BY db ASC


Query was canceled by user

Total execution time: 00:00:00.002

57.  Készítsünk listát az ügyfelek számáról születési év szerint, azon belül nem szerinti bontásban!  
    
    a. A lista megfelelően jelölve tartalmazza a részösszegeket és a végösszeget is!  
    b. Az oszlopoknak adjunk nevet értelemszerűen!

In [21]:
SELECT 
    IIF(GROUPING(SZULEV)=1, 'Összesen', cast(SZULEV as nvarchar(20))) as 'Szülév',
    (
    case GROUPING_ID(SZULEV, NEM)
        when 0 then NEM
        when 1 then 'Nemek összesen'
        when 3 then 'Összesen'
    End 
    ) as 'Nemek',
    COUNT(*) as 'db'
from Ugyfel
GROUP by rollup(SZULEV, NEM)

Query was canceled by user

Total execution time: 00:00:00.001

58. <span style="background-color: rgb(255, 255, 255); color: rgb(0, 0, 0); font-family: &quot;Open Sans&quot;, sans-serif; font-size: 14.44px;">Készítsünk listát a termékek számáról a felvitel hónapja, azon belül napja szerint csoportosítva.&nbsp;</span>  

a. A lista csak a részösszegeket és a végösszeget tartalmazza!  
b. Az oszlopoknak adjunk megfelelő nevet!  
c. Ötlet: HAVING + GROUPING\_ID fv együttes használata

In [22]:
SELECT MONTH(felvitel) AS 'hónap',
       DAY(felvitel) AS 'nap',
       COUNT(*) AS 'termékek száma'
FROM Termek
GROUP BY ROLLUP(MONTH(felvitel), DAY(felvitel))
HAVING GROUPING_ID(MONTH(felvitel), DAY(felvitel)) > 0

Query was canceled by user

Total execution time: 00:00:50.739

59. Jelenítsük meg a termékek kódja és listaára mellett a termékkategória átlagárát is!

In [23]:
 select TERMEKKOD,
        LISTAAR,
        AVG(LISTAAR) OVER (PARTITION BY KAT_ID) as 'Kategória átlagár'
from Termek

Query was canceled by user

Total execution time: 00:00:00.010

60\. Listázzuk az egyes megrendelések dátumát, a termék kódját és mennyiségét, valamint a sorszám szerinti előző 5 megrendelés átlagos mennyiségét is!

In [24]:
SELECT r.REND_DATUM,
        rt.TERMEKKOD,
        rt.MENNYISEG,
        AVG(rt.MENNYISEG) OVER(PARTITION BY rt.TERMEKKOD
                                ORDER by rt.SORSZAM
                                rows BETWEEN 5 preceding and 1 preceding)
        as 'Előző 5 megrendelés átlagos mennyisége'
from Rendeles r JOIN Rendeles_tetel rt on r.SORSZAM=rt.SORSZAM

Query was canceled by user

Total execution time: 00:00:00.031

61. Jelenítsük meg, hogy az egyes ügyfelek az adott rendelési dátumig bezárólag összesen hányszor rendeltek! Megjelenítendő a rendelés dátuma, az ügyfél login-ja és a rendelés darabszáma

In [25]:
SELECT distinct REND_DATUM, 
        [LOGIN],
        COUNT(*) OVER(PARTITION by LOGIN
                        ORDER BY REND_DATUM
                        range BETWEEN unbounded preceding and current row)
        as 'Eddigi rendelések'
from Rendeles

Query was canceled by user

Total execution time: 00:00:00.027

62\. Készítsünk sorszámozott listát nemenként az ügyfelekről! A sorszámozás szempontja az ügyfél email-címe legyen!

In [26]:
SELECT ROW_NUMBER() OVER(PARTITION by NEM
                        ORDER BY EMAIL) 
        as 'nemenkénti sorszám',
        *
from Ugyfel

Query was canceled by user

Total execution time: 00:00:00.007

63. Listázzuk a termékek kódját, megnevezését, kategória kódját, készlet mennyiségét és azt, hogy a termék a készlet alapján hányadik a kategóriájában

In [27]:
SELECT TERMEKKOD,
        MEGNEVEZES,
        KAT_ID,
        KESZLET,
        RANK() OVER(PARTITION BY KAT_ID
                    ORDER BY KESZLET desc)
        as 'Készlet szerinti helyezés'
from Termek

Query was canceled by user

Total execution time: 00:00:00.005

64. Az előző példa DENSE\_RANK() függvénnyel

In [28]:
SELECT TERMEKKOD,
        MEGNEVEZES,
        KAT_ID,
        KESZLET,
        DENSE_RANK() OVER(PARTITION BY KAT_ID
                    ORDER BY KESZLET desc)
        as 'Készlet szerinti helyezés'
from Termek

Query was canceled by user

Total execution time: 00:00:00.002

65. Listázzuk minden rendelési tétel sorszámát, a termék kódját és mennyiségét, valamint az adott termék előző rendelésének mennyiségét!

In [29]:
SELECT SORSZAM, TERMEKKOD, MENNYISEG,
LAG(MENNYISEG,1,0) OVER(PARTITION BY
TERMEKKOD ORDER BY SORSZAM)
AS 'Előző rendelési mennyiség'
FROM Rendeles_tetel

Query was canceled by user

Total execution time: 00:00:00.001

66. Listázzuk minden rendelési tétel sorszámát, a termék kódját és mennyiségét, valamint az adott termék kettővel későbbi rendelésének mennyiségét!

In [30]:
SELECT SORSZAM,
        TERMEKKOD,
        MENNYISEG,
        LEAD(MENNYISEG,2,0) OVER(PARTITION BY TERMEKKOD
                                ORDER BY SORSZAM)
from Rendeles_tetel

Query was canceled by user

Total execution time: 00:00:00.001

67.  Listázzuk az egyes ügyfelek adatait és első rendelésük dátumát! A lista ne tartalmazzon duplikált sorokat!

In [31]:
SELECT distinct u.*,
                FIRST_VALUE(r.REND_DATUM)
                OVER(PARTITION BY u.LOGIN
                    ORDER by r.REND_DATUM)
                as 'első rendelés'
from Ugyfel u JOIN Rendeles r on u.[LOGIN]=r.[LOGIN]

Query was canceled by user

Total execution time: 00:00:00

68. Soroljuk be a termékeket kategóriájukban a listaáruk alapján 5 osztályba!

In [32]:
SELECT *,
        ntile(5) OVER(PARTITION BY KAT_ID
                    order by LISTAAR)
        as 'Osztály'
from termek

Query was canceled by user

Total execution time: 00:00:00

69. <span style="color: rgb(36, 41, 47); font-family: -apple-system, BlinkMacSystemFont, &quot;Segoe UI&quot;, Helvetica, Arial, sans-serif, &quot;Apple Color Emoji&quot;, &quot;Segoe UI Emoji&quot;; font-size: 16px;">Készítsünk listát éves bontásban norbert2 azonosítójú ügyfél rendeléseinek értékéről!</span>
    
    1. A lista megfelelően jelölve tartalmazza a végösszeget is!

In [33]:
SELECT
(
    case GROUPING(YEAR(r.rend_datum))
        when 0 then CAST(YEAR(r.rend_datum) as nvarchar(4))
        when 1 then 'Összesen'
    END
) as 'Év',
sum(rt.egysegar*rt.mennyiseg)
from Ugyfel u JOIN Rendeles r on u.LOGIN=r.LOGIN
                JOIN Rendeles_tetel rt on r.sorszam=rt.sorszam
WHERE r.LOGIN='norbert2'
group BY ROLLUP(YEAR(r.rend_datum))

Query was canceled by user

Total execution time: 00:00:00

70.  Készítsünk listát szállítási dátumonként, azon belül szállítási módonként az egyes rendelések összmennyiségéről!
    1. Csak azokat a termékeket vegyük figyelembe, amelyek mennyiségi egysége db!
    2. A listát szűrjük úgy, hogy az csak a részösszeg sorokat és a végösszeget tartalmazza!

In [34]:
SELECT IIF(GROUPING(r.szall_datum)=1, 'Összesen', 
            CAST(r.szall_datum as NVARCHAR(10))) as 'Dátum',
        (
            case GROUPING_ID(r.SZALL_DATUM, r.SZALL_MOD)
                when 0 then r.SZALL_MOD
                when 1 then 'Szállítási módok összesen'
                when 3 then 'Összesen'
            End
        ) as 'Szall_mod',
        sum(rt.mennyiseg) as 'Összmennyiség'
FROM Rendeles r JOIN Rendeles_tetel rt on r.sorszam=rt.sorszam
                join termek t on rt.TERMEKKOD=t.TERMEKKOD
where t.MEGYS='db'
group by ROLLUP(r.szall_datum, r.Szall_mod)
HAVING GROUPING_ID(r.SZALL_DATUM, r.SZALL_MOD) in (1,3)

Query was canceled by user

Total execution time: 00:00:00.001

In [35]:
SELECT r.SZALL_DATUM, 
       r.SZALL_MOD, 
       SUM(rt.MENNYISEG) AS 'Összmennyiség'
FROM Rendeles_tetel rt JOIN Termek t ON rt.TERMEKKOD = t.TERMEKKOD
                       JOIN Rendeles r ON r.SORSZAM = rt.SORSZAM
WHERE t.MEGYS='db'
GROUP BY ROLLUP(r.SZALL_DATUM, r.SZALL_MOD)
HAVING GROUPING_ID(r.SZALL_DATUM, r.SZALL_MOD) IN (1,3)

Query was canceled by user

Total execution time: 00:00:00

```
71. Hány olyan ügyfél van, aki még nem rendelt semmit?

```

1. Csoportosítsuk őket nem szerint, azon belül életkor szerint!
2. A lista tartalmazza a részösszegeket és a végösszeget is!

In [36]:
SELECT u.nem,
        YEAR(GETDATE()) - u.SZULEV as 'Életkor',
        COUNT(*)
from ugyfel u left JOIN rendeles r on u.LOGIN=r.LOGIN
where r.LOGIN is NULL
GROUP BY ROLLUP( u.nem,  YEAR(GETDATE()) - u.SZULEV)

Query was canceled by user

Total execution time: 00:00:00

72. <span style="color: rgb(36, 41, 47); font-family: -apple-system, BlinkMacSystemFont, &quot;Segoe UI&quot;, Helvetica, Arial, sans-serif, &quot;Apple Color Emoji&quot;, &quot;Segoe UI Emoji&quot;; font-size: 16px;">Készítsünk listát a megrendelt termékek legkisebb és legnagyobb egységáráról szállítási dátum, azon belül szállítási mód szerinti bontásban!</span>

1. A lista csak a 2015 májusi szállításokat tartalmazza!
2. Jelenítsük meg a részösszegeket és a végösszeget is!

In [37]:
SELECT r.szall_datum,  
        r.Szall_mod,
        min(rt.egysegar) as 'legkisebb',
        MAX(rt.egysegar) as 'legnagyobb'
from Rendeles_tetel rt left JOIN rendeles r on rt.SORSZAM=r.sorszam
where YEAR(r.szall_datum)=2015 AND MONTH(r.szall_datum)=5
GROUP BY ROLLUP( r.szall_datum,  r.Szall_mod)

Query was canceled by user

Total execution time: 00:00:00

Commands completed successfully.

Total execution time: 00:00:00

Commands completed successfully.

Total execution time: 00:00:00

zhgyak 1.  

Kérdezzük le, hogy melyik ügyfél (USERNEV) hány különbözö szálláshelyen foglalt!

a. A listában azok az ügyfelek is jelenjenek meg, akiknek még nem volt foglalásuk

b. Megfelelően jelölve jelenjen meg a végösszeg is!

In [40]:
SELECT
(
    case GROUPING(v.usernev)
    when 0 then v.USERNEV
    when 1 then 'Összesen'
    end 
) as 'Ügyfél',
COUNT(distinct sz.SZALLAS_FK) as 'db'
from Vendeg v left join Foglalas f on v.usernev=f.UGYFEL_FK
                    JOIN Szoba sz on f.SZOBA_FK=sz.SZOBA_ID
GROUP BY rollup(v.usernev)

Query was canceled by user

Total execution time: 00:00:00

zhgyak 2.

Készítsünk listát, amely megjeleníti a vendégek adatait!

· Egy új oszlopban számoljuk ki a vendég életkorát (években)

· Egy másik új oszlopban határozzuk meg, hogy születési dátum szerint növekvö rendezésnél mennyi az adott ügyfél. az előtte lévö 2 ügyfél és az utána lévő 2 ügyfél átlagos életkora! Az oszlopot

nevezzük el értelemszerűen!

In [41]:
SELECT *,
       YEAR(GETDATE())-YEAR(szul_dat) as 'életkor',
        AVG( YEAR(GETDATE())-YEAR(szul_dat)) 
                OVER(
                    ORDER by SZUL_DAT  
                    rows BETWEEN 2 preceding and 2 following)
from Vendeg
ORDER by SZUL_DAT asc

Query was canceled by user

Total execution time: 00:00:00

Egészítsük ki a megkezdett lekérdezést, amely listázza azon vendégek azonosítóját és nevét, akik már legalább egyszer foglaltak, és MINDEN ESETBEN összesen két fő számára (felnőtt + gyermek

szám összege)! a. A lista ne tartalmazzon ismetlodo sorokat!

In [42]:
SELECT distinct v.USERNEV,
        v.NEV
FROM Vendeg v JOIN Foglalas f ON v.USERNEV = f.UGYFEL_FK
WHERE NOT EXISTS
(
    SELECT 1
    FROM Foglalas f2
    WHERE f2.UGYFEL_FK = v.USERNEV 
        AND f2.FELNOTT_SZAM +f2.GYERMEK_SZAM <>2
)

Query was canceled by user

Total execution time: 00:00:00

zh4 gyak 1. sorszamozzuk a vendegeket a foglalasok szama alapjan

In [45]:
SELECT 
    v.USERNEV,
    v.NEV,
    COUNT(*) AS FoglalasokSzama,
    DENSE_RANK() OVER (
        ORDER BY COUNT(*) DESC
    ) AS Sorszam
FROM Vendeg v LEFT JOIN Foglalas f ON v.USERNEV = f.UGYFEL_FK
GROUP BY v.USERNEV, v.NEV


(213 rows affected)

Total execution time: 00:00:00.031

USERNEV,NEV,FoglalasokSzama,Sorszam
,Kiss József,28,1
akos,Bíró Ákos,22,2
andi,Maródi Andrea,21,3
aladar,Dunai Aladár,20,4
agnes3,Hartyánszky Ágnes,20,4
andras41,Komjáti András,16,5
andras2,Tóth András,15,6
ARONK,Kelemen Áron,12,7
krisztian4,Czérna Krisztián,12,7
peter4,Bíró Péter,12,7


2. Listázzuk azon vendégek nevét, email-címét és felhasználói nevét,

akik egynél többször foglaltak!

· Hagyjuk ki azokat a vendégeket, akik május hónapban születtek!

In [48]:
SELECT v.NEV, 
        v.EMAIL, 
        v.USERNEV
FROM Vendeg v JOIN Foglalas f ON v.USERNEV = f.UGYFEL_FK
WHERE MONTH(v.SZUL_DAT) <> 5
GROUP BY v.NEV, v.EMAIL, v.USERNEV
HAVING COUNT(f.FOGLALAS_PK) > 1

(170 rows affected)

Total execution time: 00:00:00.108

NEV,EMAIL,USERNEV
Kiss Ádám,ádám.kiss@mail.hu,adam1
Barkóci Ádám,adam3@gmail.com,adam3
Bieniek Ádám,ádám.bieniek@mail.hu,adam4
Lengyel Ágnes,agnes@gmail.com,agnes
Hartyánszky Ágnes,agnes3@gmail.com,agnes3
Horváth Ágnes,AGNESH@gmail.com,AGNESH
Kovács Ágnes,AGNESK@gmail.com,AGNESK
Bíró Ákos,ákos.bíró@mail.hu,akos
Dunai Aladár,aladár.dunai@mail.hu,aladar
Bagóczki Alexandra,alexandra.bagóczki@mail.hu,alexandra


3. Listázzuk azon vendegek adatait, akik a legtobb férőhelyes szobát

(vagy szobákat) már lefoglalták!

· Csak klímás szoba jöhet számításba

In [49]:
SELECT DISTINCT v.*
FROM Vendeg v JOIN Foglalas f ON v.USERNEV = f.UGYFEL_FK
                JOIN Szoba s ON f.SZOBA_FK = s.SZOBA_ID
WHERE s.KLIMAS = 'I'
    AND s.FEROHELY = (
        SELECT MAX(FEROHELY)
        FROM Szoba
        WHERE KLIMAS = 'I'
    );

(3 rows affected)

Total execution time: 00:00:00.032

USERNEV,NEV,EMAIL,SZAML_CIM,SZUL_DAT
JOZSEFG,Gyuris József,józsef.gyuris@mail.hu,2660 Balassagyarmat Petőfi utca 1/2.,1975-05-26
PETERB,Berendi Péter,péter.berendi@mail.hu,3980 Sátoraljaújhely Vasút utca 4/10.,1969-01-01
tunde,Turcsik Tünde,tunde@gmail.com,7130 Tolna Fő út 122.,1974-02-12
